# 🔄 Fair Student v2 — Retraining Notebook

**Goal:** Retrain the fairness-aware MobileNetV2 student model with **increased fairness weight (β=0.4)** to reduce the **FFPR gap below 0.12**.

## Background

| Metric | v1 Result (β=0.2) | Target |
|--------|-------------------|--------|
| **FFPR Gap** | 0.1995 | **≤ 0.12** |
| AUC | 0.9497 | ≥ 0.95 |
| Model Size | 10.2 MB | ≤ 15 MB |

## Loss Function Change
```
v1: L = 0.7·L_distill + 0.2·L_fairness + 0.1·L_cls
v2: L = 0.5·L_distill + 0.4·L_fairness + 0.1·L_cls   ← KEY CHANGE
```

## Architecture
- **Teacher**: XceptionNet (~88MB, frozen during distillation)
- **Student**: MobileNetV2 (~10.2 MB, trained from ImageNet weights)
- **Fairness**: Pairwise accuracy-gap loss across 8 demographic groups (Gender × 4 Races)

**⏱ Estimated training time: 12–24 hours on RTX 4050**

---
## 📦 Cell 1 — Install & Upgrade Dependencies

In [1]:
# ================================================================
# Cell 1: Install / Upgrade all required packages
# Run this ONCE at the start of a new environment or after updates
# ================================================================
import subprocess, sys

def pip_install(package, upgrade=False):
    """Install a package using pip."""
    cmd = [sys.executable, "-m", "pip", "install"]
    if upgrade:
        cmd.append("--upgrade")
    cmd.append(package)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✅ {package}")
    else:
        print(f"  ❌ {package}: {result.stderr.strip()[-200:]}")

print("============================================================")
print(" Installing / Verifying Dependencies")
print("============================================================")

# Core deep learning
pip_install("torch>=2.0.0")
pip_install("torchvision>=0.15.0")
pip_install("torchaudio")  # needed for CUDA context on some setups

# Model library (for timm XceptionNet support)
pip_install("timm>=0.9.0")

# Face detection (MTCNN — used during preprocessing)
pip_install("facenet-pytorch>=2.5.0")

# Image processing
pip_install("opencv-python>=4.8.0")
pip_install("Pillow>=10.0.0")

# Data & metrics
pip_install("pandas>=2.0.0")
pip_install("numpy>=1.24.0")
pip_install("scikit-learn>=1.3.0")

# Visualization
pip_install("matplotlib>=3.7.0")
pip_install("seaborn>=0.12.0")

# Progress bar
pip_install("tqdm>=4.65.0")

# GradCAM explainability
pip_install("grad-cam>=1.4.0")

# Streamlit demo (optional for this notebook)
pip_install("streamlit>=1.28.0")

# Jupyter display utilities
pip_install("ipywidgets")

print("\n✅ All packages processed.")

 Installing / Verifying Dependencies
  ✅ torch>=2.0.0
  ✅ torchvision>=0.15.0
  ✅ torchaudio
  ✅ timm>=0.9.0
  ✅ facenet-pytorch>=2.5.0
  ✅ opencv-python>=4.8.0
  ✅ Pillow>=10.0.0
  ✅ pandas>=2.0.0
  ✅ numpy>=1.24.0
  ✅ scikit-learn>=1.3.0
  ✅ matplotlib>=3.7.0
  ✅ seaborn>=0.12.0
  ✅ tqdm>=4.65.0
  ✅ grad-cam>=1.4.0
  ✅ streamlit>=1.28.0
  ✅ ipywidgets

✅ All packages processed.


---
## 🔍 Cell 2 — Environment & GPU Validation

In [2]:
# ================================================================
# Cell 2: Verify GPU, CUDA, package versions, and project paths
# ================================================================
import os, sys
import torch
import torchvision
import numpy as np
import pandas as pd

# -- Ensure project root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.dirname("."))  # adjust if needed
NOTEBOOK_DIR = os.path.abspath(".")
# Try to auto-detect: look for src/ folder
if os.path.exists(os.path.join(NOTEBOOK_DIR, "src")):
    PROJECT_ROOT = NOTEBOOK_DIR
elif os.path.exists(os.path.join(os.path.dirname(NOTEBOOK_DIR), "src")):
    PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("============================================================")
print(" Environment Validation")
print("============================================================")
print(f"  Python:       {sys.version.split()[0]}")
print(f"  PyTorch:      {torch.__version__}")
print(f"  Torchvision:  {torchvision.__version__}")
print(f"  NumPy:        {np.__version__}")
print(f"  Pandas:       {pd.__version__}")
print(f"  Project root: {PROJECT_ROOT}")

print()
print("============================================================")
print(" GPU Status")
print("============================================================")
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    vram_free  = (torch.cuda.get_device_properties(0).total_memory -
                  torch.cuda.memory_reserved(0)) / (1024**3)
    print(f"  ✅ GPU: {gpu_name}")
    print(f"   CUDA version:  {torch.version.cuda}")
    print(f"   VRAM total:    {vram_total:.1f} GB")
    print(f"   VRAM free:     {vram_free:.1f} GB")
    print(f"   cuDNN version: {torch.backends.cudnn.version()}")
    print(f"   AMP support:   ✅ (fp16 enabled)")
else:
    device = torch.device("cpu")
    print("  ⚠️  No GPU detected — training will run on CPU (very slow!)")
    print("      Install CUDA-enabled PyTorch: https://pytorch.org/get-started")

print()
print("============================================================")
print(" Project Paths Validation")
print("============================================================")
from src.config import (
    MODELS_DIR, SPLITS_DIR, RESULTS_DIR, FIGURES_DIR,
    DEVICE as CFG_DEVICE
)

required_paths = {
    "Train CSV":          os.path.join(SPLITS_DIR, "train.csv"),
    "Val CSV":            os.path.join(SPLITS_DIR, "val.csv"),
    "Test CSV":           os.path.join(SPLITS_DIR, "test.csv"),
    "Teacher Checkpoint": os.path.join(MODELS_DIR, "xception_teacher_best.pth"),
    "v1 Backup Dir":      os.path.join(MODELS_DIR, "v1_backup"),
}

all_ok = True
for name, path in required_paths.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    if not exists and name in ["Train CSV", "Val CSV", "Teacher Checkpoint"]:
        all_ok = False
    size_info = ""
    if exists and os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024**2)
        size_info = f"  ({size_mb:.1f} MB)"
    print(f"  {status} {name}: {path}{size_info}")

print()
if all_ok:
    print("  ✅ All required files present — ready to train!")
else:
    print("  ❌ Some required files are missing. Resolve before proceeding.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\Student\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Student\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "d:\M3\DeepFake_Research\.venv\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\M3\DeepFake_Research\.venv\lib\site-packages\traitlets\config\application.py", line 1075, in lau

 Environment Validation
  Python:       3.10.8
  PyTorch:      2.2.2+cpu
  Torchvision:  0.17.2+cpu
  NumPy:        2.2.6
  Pandas:       2.3.3
  Project root: d:\M3\DeepFake_Research

 GPU Status
  ⚠️  No GPU detected — training will run on CPU (very slow!)
      Install CUDA-enabled PyTorch: https://pytorch.org/get-started

 Project Paths Validation
  ✅ Train CSV: d:\M3\DeepFake_Research\outputs\splits\train.csv  (57.5 MB)
  ✅ Val CSV: d:\M3\DeepFake_Research\outputs\splits\val.csv  (12.3 MB)
  ✅ Test CSV: d:\M3\DeepFake_Research\outputs\splits\test.csv  (12.3 MB)
  ✅ Teacher Checkpoint: d:\M3\DeepFake_Research\outputs\models\xception_teacher_best.pth  (250.6 MB)
  ❌ v1 Backup Dir: d:\M3\DeepFake_Research\outputs\models\v1_backup

  ✅ All required files present — ready to train!


---
## ⚙️ Cell 3 — Configure Hyperparameters

In [3]:
# ================================================================
# Cell 3: v2 Retraining Hyperparameters
# Modify these to experiment with different fairness pressures.
# ================================================================
import torch

# ── Loss weights ─────────────────────────────────────────────────
V2_ALPHA       = 0.5     # Distillation loss weight (was 0.7 in v1)
V2_BETA        = 0.4     # Fairness loss weight     (was 0.2 in v1) ← KEY CHANGE
V2_GAMMA       = 0.1     # Classification loss weight (unchanged)

# ── Training params ──────────────────────────────────────────────
V2_TEMPERATURE = 4.0     # Distillation temperature (unchanged)
V2_EPOCHS      = 50      # Max epochs (early stopping will trigger earlier)
V2_BATCH_SIZE  = 512      # 32 keeps teacher+student within 6 GB VRAM
V2_LR          = 1e-4    # AdamW learning rate
V2_WEIGHT_DECAY = 1e-5   # AdamW weight decay
V2_MODEL_NAME  = "fair_student"  # Same name so evaluation scripts work

# ── Curriculum learning (optional) ───────────────────────────────
USE_CURRICULUM = False   # Set True to gradually ramp beta from beta/4 → V2_BETA

# ── Device ───────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("============================================================")
print(" v2 Retraining Configuration")
print("============================================================")
print(f"  Loss: L = {V2_ALPHA}·L_distill + {V2_BETA}·L_fairness + {V2_GAMMA}·L_cls")
print(f"  Temperature:      {V2_TEMPERATURE}")
print(f"  Max epochs:       {V2_EPOCHS}")
print(f"  Batch size:       {V2_BATCH_SIZE}")
print(f"  Learning rate:    {V2_LR}")
print(f"  Weight decay:     {V2_WEIGHT_DECAY}")
print(f"  Curriculum beta:  {USE_CURRICULUM}")
print(f"  Device:           {DEVICE}")
print()
print("  --- v1 vs v2 Comparison ---")
print(f"  {'Parameter':20s}  {'v1':>8s}  {'v2':>8s}  {'Change':>8s}")
print(f"  {'-'*20}  {'-'*8}  {'-'*8}  {'-'*8}")
print(f"  {'Alpha (distill)':20s}  {'0.7':>8s}  {V2_ALPHA:>8.1f}  {'↓ -0.2':>8s}")
print(f"  {'Beta (fairness)':20s}  {'0.2':>8s}  {V2_BETA:>8.1f}  {'↑ +0.2':>8s}")
print(f"  {'Gamma (cls)':20s}  {'0.1':>8s}  {V2_GAMMA:>8.1f}  {'= same':>8s}")
print(f"  {'Epochs':20s}  {'50':>8s}  {V2_EPOCHS:>8d}  {'= same':>8s}")
print(f"  {'Batch size':20s}  {'32':>8s}  {V2_BATCH_SIZE:>8d}  {'= same':>8s}")





---
## 💾 Cell 4 — Backup v1 Checkpoints & Initialize Models

In [4]:
# ================================================================
# Cell 4: Backup v1 checkpoints, load teacher, create student
# ================================================================
import os, sys, shutil, torch

from src.config import DEVICE, MODELS_DIR, SPLITS_DIR, DISTILL_LR, DISTILL_WEIGHT_DECAY, PATIENCE
from src.models.xception   import build_teacher
from src.models.mobilenetv2 import build_student

BACKUP_DIR          = os.path.join(MODELS_DIR, "v1_backup")
TEACHER_CHECKPOINT  = os.path.join(MODELS_DIR, "xception_teacher_best.pth")
FAIR_STUDENT_FILES  = [
    "fair_student_best_auc.pth",
    "fair_student_best_fair.pth",
    "fair_student_final.pth",
]

# ── Step 1: Backup v1 checkpoints ─────────────────────────────
print("--- Step 1: Backing up v1 checkpoints ---")
os.makedirs(BACKUP_DIR, exist_ok=True)
backed_up = 0
for fname in FAIR_STUDENT_FILES:
    src = os.path.join(MODELS_DIR, fname)
    dst = os.path.join(BACKUP_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(src) / (1024**2)
        print(f"  ✅ Backed up: {fname} ({size_mb:.1f} MB)")
        backed_up += 1
if backed_up == 0:
    print("  ℹ️  No existing v1 checkpoints to back up (fresh start).")

# ── Step 2: Delete old checkpoints ────────────────────────────
print("\n--- Step 2: Removing old checkpoints (starting fresh) ---")
for fname in FAIR_STUDENT_FILES:
    path = os.path.join(MODELS_DIR, fname)
    if os.path.exists(path):
        os.remove(path)
        print(f"  🗑  Deleted: {fname}")

# ── Step 3: Load teacher model ─────────────────────────────────
print("\n--- Step 3: Loading teacher model (XceptionNet) ---")
if not os.path.exists(TEACHER_CHECKPOINT):
    raise FileNotFoundError(
        f"Teacher checkpoint not found: {TEACHER_CHECKPOINT}\n"
        "Train the teacher first (Cell 8 in the main deepfake_training.ipynb)."
    )

teacher = build_teacher(pretrained=False, device=DEVICE)
ckpt = torch.load(TEACHER_CHECKPOINT, map_location=DEVICE, weights_only=False)
teacher.load_state_dict(ckpt["model_state_dict"])
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print(f"  ✅ Teacher loaded  — val AUC: {ckpt.get('val_auc', 'N/A')}")

# ── Step 4: Create fresh student model ────────────────────────
print("\n--- Step 4: Creating fresh MobileNetV2 student ---")
student = build_student(pretrained=True, device=DEVICE)
print(f"  ✅ Student created (ImageNet weights, fresh classifier head)")

print("\n  Ready to train! ✅")

--- Step 1: Backing up v1 checkpoints ---
  ✅ Backed up: fair_student_best_auc.pth (29.6 MB)
  ✅ Backed up: fair_student_best_fair.pth (10.0 MB)
  ✅ Backed up: fair_student_final.pth (10.0 MB)

--- Step 2: Removing old checkpoints (starting fresh) ---
  🗑  Deleted: fair_student_best_auc.pth
  🗑  Deleted: fair_student_best_fair.pth
  🗑  Deleted: fair_student_final.pth

--- Step 3: Loading teacher model (XceptionNet) ---
[Teacher] XceptionNet loaded:
  Total params:     21.9M
  Trainable params: 21.9M
  Model size:       83.6 MB


d:\M3\DeepFake_Research\.venv\lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


  ✅ Teacher loaded  — val AUC: 0.9670840116434661

--- Step 4: Creating fresh MobileNetV2 student ---
[Student] MobileNetV2 loaded:
  Total params:     2.6M
  Trainable params: 2.6M
  Model size:       9.9 MB
  ✅ Student created (ImageNet weights, fresh classifier head)

  Ready to train! ✅


---
## 📊 Cell 5 — Load Data & Setup Optimizer

In [5]:
loaders = create_dataloaders(TRAIN_CSV, VAL_CSV, batch_size=V2_BATCH_SIZE, num_workers=12)
train_loader = loaders['train']
val_loader   = loaders['val']

assert not isinstance(train_loader, str), 'Unpacking error: train_loader is a string!'






---
## 🚀 Cell 6 — Start Retraining (Fair Distillation v2)

> **⏱ This cell takes 12–24 hours.** Training will print per-epoch results.
> Model checkpoints are auto-saved to `outputs/models/`.

In [6]:
# ================================================================
# Cell 6: Run fair distillation training (v2, beta=0.4)
# ================================================================
import time
from src.training.train_distill import train_fair_distillation

print("=" * 65)
print("  FAIR STUDENT RETRAINING v2")
print(f"  Loss: L = {V2_ALPHA}·Ld + {V2_BETA}·Lf + {V2_GAMMA}·Lc")
print("=" * 65)

start_time = time.time()

results = train_fair_distillation(
    student=student,
    teacher=teacher,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=V2_EPOCHS,
    alpha=V2_ALPHA,
    beta=V2_BETA,
    gamma=V2_GAMMA,
    temperature=V2_TEMPERATURE,
    curriculum=USE_CURRICULUM,
    model_name=V2_MODEL_NAME,
    device=DEVICE,
)

elapsed  = time.time() - start_time
hours    = elapsed / 3600

print("\n" + "=" * 65)
print("  TRAINING COMPLETE")
print("=" * 65)
print(f"  Total time:        {hours:.1f} hours")
print(f"  Best Val AUC:      {results['best_auc']:.4f}")
print(f"  Best Acc Gap:      {results['best_fairness_gap']:.4f}")
print(f"\n  Checkpoints saved to: outputs/models/")
print(f"  v1 backup at:         outputs/models/v1_backup/")

ImportError: cannot import name 'GradScaler' from 'torch.amp' (d:\M3\DeepFake_Research\.venv\lib\site-packages\torch\amp\__init__.py)

---
## 📈 Cell 7 — Plot Training History

In [ ]:
# ================================================================
# Cell 7: Visualize training curves after training completes
# ================================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from src.config import FIGURES_DIR

history = results["history"]
epochs  = range(1, len(history["train"]) + 1)

fig = plt.figure(figsize=(16, 10))
fig.suptitle("Fair Student v2 — Training History (β=0.4)", fontsize=14, fontweight="bold")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── AUC ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, [e["auc"] for e in history["train"]], label="Train", color="royalblue")
ax1.plot(epochs, [e["auc"] for e in history["val"]],   label="Val",   color="tomato")
ax1.axhline(0.95, linestyle="--", color="green", alpha=0.6, label="Target 0.95")
ax1.set_title("AUC"); ax1.set_xlabel("Epoch"); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# ── Accuracy ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, [e["accuracy"] for e in history["train"]], label="Train", color="royalblue")
ax2.plot(epochs, [e["accuracy"] for e in history["val"]],   label="Val",   color="tomato")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# ── Acc Fairness Gap ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
if "accuracy_gap" in history["val"][0]:
    ax3.plot(epochs, [e["accuracy_gap"] for e in history["val"]], color="darkorange", label="Val Acc Gap")
    ax3.axhline(0.08, linestyle="--", color="green", alpha=0.6, label="Target ≤0.08")
    ax3.set_title("Accuracy Gap (Fairness)"); ax3.set_xlabel("Epoch")
    ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

# ── Loss components ──────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(epochs, [e["loss"] for e in history["train"]], label="Train Total", color="royalblue")
ax4.plot(epochs, [e["loss"] for e in history["val"]],   label="Val Total",   color="tomato")
ax4.set_title("Total Loss"); ax4.set_xlabel("Epoch"); ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(gs[1, 1])
if "distill_loss" in history["train"][0]:
    ax5.plot(epochs, [e["distill_loss"]  for e in history["train"]], label="Distill",  color="steelblue")
    ax5.plot(epochs, [e["fairness_loss"] for e in history["train"]], label="Fairness", color="darkorange")
    ax5.plot(epochs, [e["cls_loss"]      for e in history["train"]], label="Cls",      color="mediumpurple")
    ax5.set_title("Train Loss Components"); ax5.set_xlabel("Epoch")
    ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

# ── Per-group val accuracy (last epoch) ──────────────────────
ax6 = fig.add_subplot(gs[1, 2])
last_val = history["val"][-1]
if "group_accuracies" in last_val and last_val["group_accuracies"]:
    gnames = sorted(last_val["group_accuracies"].keys())
    gvals  = [last_val["group_accuracies"][g] for g in gnames]
    colors = ["tomato" if v == min(gvals) else ("limegreen" if v == max(gvals) else "steelblue") for v in gvals]
    bars = ax6.bar(range(len(gnames)), gvals, color=colors)
    ax6.set_xticks(range(len(gnames)))
    ax6.set_xticklabels([g.replace("-", "\n") for g in gnames], fontsize=7)
    ax6.axhline(min(gvals), linestyle="--", color="red",   alpha=0.5)
    ax6.axhline(max(gvals), linestyle="--", color="green", alpha=0.5)
    ax6.set_title("Per-Group Val Accuracy\n(Last Epoch)"); ax6.set_ylim(0.8, 1.0)
    ax6.grid(True, axis="y", alpha=0.3)

plt.savefig(os.path.join(FIGURES_DIR, "retrain_v2_history.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"  ✅ Saved → {FIGURES_DIR}/retrain_v2_history.png")

---
## 🏆 Cell 8 — Evaluation: Test Set + Cross-Dataset + Fairness Metrics

In [ ]:
# ================================================================
# Cell 8: Load best student checkpoint and evaluate on test set
# ================================================================
import torch, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

from src.config import DEVICE, MODELS_DIR, SPLITS_DIR, RESULTS_DIR, NUM_GROUPS, IDX_TO_GROUP
from src.data.dataset import DeepfakeDataset
from src.models.mobilenetv2 import build_student
from src.evaluation.fairness_metrics import compute_group_metrics, compute_fairness_metrics

CHECKPOINT = os.path.join(MODELS_DIR, "fair_student_best_auc.pth")
BATCH_SIZE  = 256

# Load best model
print(f"--- Loading best v2 checkpoint ---")
if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}. Run training first.")

eval_model = build_student(pretrained=False, device=DEVICE)
ckpt = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
eval_model.load_state_dict(ckpt["model_state_dict"])
eval_model.eval()
print(f"  ✅ Model loaded (saved at epoch {ckpt.get('epoch', '?')}, "
      f"val AUC: {ckpt.get('val_auc', '?'):.4f})")


def get_preds(model, csv_path, split="test", batch_size=64):
    """Run inference on a CSV split and return (probs, labels, groups)."""
    ds = DeepfakeDataset(csv_path, split=split)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=12)
    all_probs, all_labels, all_groups = [], [], []
    with torch.no_grad():
        for images, labels, groups in tqdm(dl, desc=f"  Inference [{split}]", leave=False):
            probs = torch.sigmoid(model(images.to(DEVICE))).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
            all_groups.append(groups.numpy())
    return (np.concatenate(all_probs),
            np.concatenate(all_labels),
            np.concatenate(all_groups))


def evaluate_split(csv_path, name):
    """Evaluate a dataset split and print fairness metrics."""
    if not os.path.exists(csv_path):
        print(f"  ⚠️  {name}: CSV not found → {csv_path}")
        return None

    probs, labels, groups = get_preds(eval_model, csv_path, batch_size=BATCH_SIZE)
    preds = (probs > 0.5).astype(int)
    acc   = accuracy_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = float("nan")

    gm = compute_group_metrics(labels, preds, probs, groups)
    fm = compute_fairness_metrics(gm)

    print(f"\n{'='*65}")
    print(f"  {name}")
    print(f"{'='*65}")
    print(f"  Overall Accuracy:  {acc:.4f}")
    print(f"  Overall AUC:       {auc:.4f}")
    print()

    targets = {"FFPR_gap": 0.12, "FEO_gap": 0.10, "FDP_gap": 0.10, "FOAE_gap": 0.08}
    for metric, target in targets.items():
        val = fm.get(metric, float("nan"))
        status = "✅" if val <= target else "⚠️"
        print(f"  {metric:14s}: {val:.4f}  (target ≤ {target})  {status}")

    print("\n  Per-Group Performance:")
    print(f"  {'Group':22s}  {'Acc':>8s}  {'FPR':>8s}  {'FNR':>8s}")
    print(f"  {'-'*22}  {'-'*8}  {'-'*8}  {'-'*8}")
    for g_name, m in sorted(gm.items()):
        g_acc = m.get("accuracy", float("nan"))
        g_fpr = m.get("fpr", float("nan"))
        g_fnr = m.get("fnr", float("nan"))
        print(f"  {g_name:22s}  {g_acc:8.4f}  {g_fpr:8.4f}  {g_fnr:8.4f}")

    return {"accuracy": acc, "auc": auc, **fm, "group_metrics": gm}


# ── Evaluate ─────────────────────────────────────────────────────
test_results = evaluate_split(os.path.join(SPLITS_DIR, "test.csv"), "FF++ TEST SET")
celebdf_res  = evaluate_split(os.path.join(SPLITS_DIR, "Celeb-DF_test.csv"), "CROSS-DATASET: Celeb-DF")
dfd_res      = evaluate_split(os.path.join(SPLITS_DIR, "DFD_test.csv"), "CROSS-DATASET: DFD")



---
## 📊 Cell 9 — Compare v1 vs v2 Results

In [ ]:
# ================================================================
# Cell 9: Side-by-side v1 vs v2 comparison & save results CSV
# ================================================================
import pandas as pd
from src.config import RESULTS_DIR

# Known v1 results from RETRAINING_GUIDE.md
v1_known = {
    "FFPR_gap": 0.1995,
    "AUC":      0.9497,
    "Accuracy": 0.9189,
    "FOAE_gap": 0.0702,
}

print("=" * 65)
print("  v1 (β=0.2) vs v2 (β=0.4) — FF++ Test Set")
print("=" * 65)

targets = {"FFPR_gap": 0.12, "FEO_gap": 0.10, "FDP_gap": 0.10, "FOAE_gap": 0.08}

if test_results:
    metrics = [
        ("AUC",       v1_known["AUC"],       test_results.get("auc",      float("nan")), 0.95,  False),
        ("Accuracy",  v1_known["Accuracy"],   test_results.get("accuracy", float("nan")), 0.91,  False),
        ("FFPR_gap",  v1_known["FFPR_gap"],   test_results.get("FFPR_gap", float("nan")), 0.12,  True),
        ("FOAE_gap",  v1_known["FOAE_gap"],   test_results.get("FOAE_gap", float("nan")), 0.08,  True),
    ]

    print(f"  {'Metric':14s}  {'v1 (β=0.2)':>12s}  {'v2 (β=0.4)':>12s}  {'Target':>8s}  {'Status':>6s}")
    print(f"  {'-'*14}  {'-'*12}  {'-'*12}  {'-'*8}  {'-'*6}")
    for name, v1, v2, tgt, lower_is_better in metrics:
        delta = v2 - v1
        sign  = "+" if delta >= 0 else ""
        met   = (v2 <= tgt) if lower_is_better else (v2 >= tgt)
        icon  = "✅" if met else "⚠️"
        print(f"  {name:14s}  {v1:12.4f}  {v2:12.4f}  {tgt:8.3f}  {icon}")

# Save comparison to CSV
rows = []
all_res = {
    "FF++ Test":  test_results,
    "Celeb-DF":   celebdf_res,
    "DFD":        dfd_res,
}
for ds_name, res in all_res.items():
    if not res:
        continue
    rows.append({
        "Dataset":   ds_name,
        "Model":     "fair_student_v2 (β=0.4)",
        "AUC":       f"{res.get('auc', float('nan')):.4f}",
        "Accuracy":  f"{res.get('accuracy', float('nan')):.4f}",
        "FFPR_gap":  f"{res.get('FFPR_gap', float('nan')):.4f}",
        "FEO_gap":   f"{res.get('FEO_gap', float('nan')):.4f}",
        "FDP_gap":   f"{res.get('FDP_gap', float('nan')):.4f}",
        "FOAE_gap":  f"{res.get('FOAE_gap', float('nan')):.4f}",
    })

out_path = os.path.join(RESULTS_DIR, "retrain_v2_evaluation.csv")
pd.DataFrame(rows).to_csv(out_path, index=False)
print(f"\n  ✅ Results saved → {out_path}")

---
## 🔬 Cell 10 — (Optional) If FFPR Gap Still > 0.12: Next Steps

In [ ]:
# ================================================================
# Cell 10: Escalation options if beta=0.4 still doesn't meet target
# ================================================================

print("============================================================")
print(" If FFPR gap > 0.12 after v2 training, try:")
print("============================================================")
options = [
    ("Option A", "β=0.6",         "Max fairness pressure",              "V2_BETA=0.6, V2_ALPHA=0.3"),
    ("Option B", "Curriculum β",  "Gradual ramp-up (stable training)",  "USE_CURRICULUM=True"),
    ("Option C", "100 epochs",    "More time to converge",              "V2_EPOCHS=100"),
    ("Option D", "FFPR loss",     "Directly penalize FPR disparity",    "Custom loss in fairness_loss.py"),
]
for opt, name, desc, change in options:
    print(f"\n  [{opt}] {name}: {desc}")
    print(f"     → Set: {change}  then re-run Cell 3-6")

print()
print("  Restore v1 checkpoints:")
print("    Copy-Item 'outputs\\models\\v1_backup\\*' 'outputs\\models\\' -Force")

# Show current FFPR gap if evaluation has run
if test_results and "FFPR_gap" in test_results:
    gap = test_results["FFPR_gap"]
    status = "✅ TARGET MET" if gap <= 0.12 else f"⚠️  Still above target (gap={gap:.4f})"
    print(f"\n  Current FFPR gap: {gap:.4f}  → {status}")
else:
    print("\n  ℹ️  Run Cell 8 first to get evaluation results.")

---
## 💡 Cell 11 — Quick Model Size & Inference Speed Check

In [ ]:
# ================================================================
# Cell 11: Verify model meets size and speed requirements
# ================================================================
import time, torch
from src.config import TARGET_MODEL_SIZE_MB, TARGET_INFERENCE_MS

print("============================================================")
print(" Model Size & Inference Speed")
print("============================================================")

# Parameter count
total_params  = sum(p.numel() for p in eval_model.parameters())
train_params  = sum(p.numel() for p in eval_model.parameters() if p.requires_grad)
param_size_mb = sum(p.nelement() * p.element_size() for p in eval_model.parameters()) / (1024**2)
buf_size_mb   = sum(b.nelement() * b.element_size() for b in eval_model.buffers()) / (1024**2)
total_size_mb = param_size_mb + buf_size_mb

print(f"  Total params:     {total_params / 1e6:.2f}M")
print(f"  Trainable params: {train_params / 1e6:.2f}M")
size_status = "✅" if total_size_mb <= TARGET_MODEL_SIZE_MB else "⚠️"
print(f"  Model size:       {total_size_mb:.1f} MB  (target ≤ {TARGET_MODEL_SIZE_MB} MB)  {size_status}")

# Inference speed benchmark
print()
eval_model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=DEVICE)

# Warm-up
with torch.no_grad():
    for _ in range(10):
        _ = eval_model(dummy_input)

# Benchmark
N_ITERS = 100
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_ITERS):
        _ = eval_model(dummy_input)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - t0) * 1000 / N_ITERS

speed_status = "✅" if elapsed_ms <= TARGET_INFERENCE_MS else "⚠️"
print(f"  Inference speed:  {elapsed_ms:.1f} ms/image  (target ≤ {TARGET_INFERENCE_MS} ms)  {speed_status}")

print()
if total_size_mb <= TARGET_MODEL_SIZE_MB and elapsed_ms <= TARGET_INFERENCE_MS:
    print("  ✅ Model meets all deployment requirements!")
else:
    print("  ⚠️  Some requirements not met. Review above.")

---
## 📝 Summary

| Step | Cell | Description |
|------|------|-------------|
| Install deps | 1 | Install all required packages |
| Verify env | 2 | GPU check, path validation |
| Configure | 3 | Set β=0.4 hyperparameters |
| Init models | 4 | Backup v1, load teacher, create student |
| Setup data | 5 | DataLoaders + optimizer |
| **Train** | **6** | **Fair distillation (12-24h)** |
| Visualize | 7 | Training curves |
| Evaluate | 8 | Test set + cross-dataset |
| Compare | 9 | v1 vs v2 metrics |
| Next steps | 10 | Escalation options if target missed |
| Speed check | 11 | Model size + inference ms |